# Final Project: Predicting Satellite Images with CNN and PyTorch

Student ID: **K12443705**


In [ ]:
# Imports, paths, and reproducibility
from pathlib import Path
import os
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

# Paths (relative to this notebook location)
BASE_DIR = Path('.').resolve()
DATA_DIR = BASE_DIR / 'data'
TEST_DIR = BASE_DIR / 'public_test_data'
ASSETS_DIR = BASE_DIR / 'assets'
PLOTS_DIR = ASSETS_DIR / 'plots'
WEIGHTS_DIR = ASSETS_DIR / 'weights'
APP_DIR = BASE_DIR / 'app'
SUBMISSION_PATH = BASE_DIR / 'submission.csv'

for p in [PLOTS_DIR, WEIGHTS_DIR, APP_DIR]:
    p.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)


In [ ]:
# Preprocessing
CLASSES = [
    'AnnualCrop','Forest','HerbaceousVegetation','Highway','Industrial',
    'Pasture','PermanentCrop','Residential','River','SeaLake'
]
IMG_EXTS = {'.jpg', '.jpeg', '.png'}


def preprocess(data_folder: str) -> tuple[pd.DataFrame, dict]:
    data_path = Path(data_folder)
    class_names = sorted([p.name for p in data_path.iterdir() if p.is_dir()])
    class_to_idx = {name: idx for idx, name in enumerate(class_names)}
    idx_to_class = {idx: name for name, idx in class_to_idx.items()}

    records = []
    for class_name in class_names:
        folder = data_path / class_name
        for img_path in sorted(folder.iterdir()):
            if img_path.is_file() and img_path.suffix.lower() in IMG_EXTS:
                records.append({
                    'folder': str(folder),
                    'file_name': img_path.name,
                    'label': class_to_idx[class_name],
                })

    df = pd.DataFrame(records)
    return df, idx_to_class


df_all, idx_to_class = preprocess(str(DATA_DIR))
class_to_idx = {v: k for k, v in idx_to_class.items()}

train_df, val_df = train_test_split(
    df_all,
    test_size=0.2,
    random_state=SEED,
    stratify=df_all['label'],
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print('Total:', len(df_all), 'Train:', len(train_df), 'Val:', len(val_df))
print('Label mapping:', idx_to_class)


In [ ]:
# EDA functions
sns.set_style('whitegrid')


def _read_img(row):
    img_path = Path(row['folder']) / row['file_name']
    return np.array(Image.open(img_path).convert('RGB'))


def show_samples(df: pd.DataFrame, num_samples: int = 5):
    sample_df = df.sample(n=min(num_samples, len(df)), random_state=SEED)
    fig, axes = plt.subplots(1, len(sample_df), figsize=(3 * len(sample_df), 3))
    if len(sample_df) == 1:
        axes = [axes]
    for ax, (_, row) in zip(axes, sample_df.iterrows()):
        img = _read_img(row)
        label_name = idx_to_class[int(row['label'])]
        ax.imshow(img)
        ax.set_title(f'Label: {label_name}')
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'random_samples.png', dpi=200, bbox_inches='tight')
    plt.show()


def average_pixel_plot(df: pd.DataFrame):
    r_vals, g_vals, b_vals = [], [], []
    for _, row in df.iterrows():
        img = _read_img(row)
        r_vals.append(img[:, :, 0].mean())
        g_vals.append(img[:, :, 1].mean())
        b_vals.append(img[:, :, 2].mean())

    plt.figure(figsize=(8, 5))
    plt.hist(r_vals, bins=30, alpha=0.5, color='red', label='Red')
    plt.hist(g_vals, bins=30, alpha=0.5, color='green', label='Green')
    plt.hist(b_vals, bins=30, alpha=0.5, color='blue', label='Blue')
    plt.title('Distribution of Average Pixel Values')
    plt.xlabel('Average Pixel Value')
    plt.ylabel('Frequency')
    plt.legend()
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'average_pixel_distribution.png', dpi=200, bbox_inches='tight')
    plt.show()


def average_brightness_per_class(df: pd.DataFrame):
    rows = []
    for _, row in df.iterrows():
        img = _read_img(row)
        brightness = img.mean()
        rows.append({'class': idx_to_class[int(row['label'])], 'brightness': brightness})

    bdf = pd.DataFrame(rows)
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=bdf, x='class', y='brightness')
    plt.xticks(rotation=45, ha='right')
    plt.title('Average Brightness Distribution per Class')
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'average_brightness.png', dpi=200, bbox_inches='tight')
    plt.show()

# Run EDA
show_samples(train_df, num_samples=5)
average_pixel_plot(train_df)
average_brightness_per_class(train_df)


In [ ]:
# Dataset and DataLoaders
class SatelliteDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = Path(row['folder']) / row['file_name']
        image = Image.open(img_path).convert('RGB')
        label = int(row['label'])

        if self.transform:
            image = self.transform(image)
        return image, label


train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = SatelliteDataset(train_df, transform=train_transform)
val_dataset = SatelliteDataset(val_df, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2)

print('Train batches:', len(train_loader), 'Val batches:', len(val_loader))


In [ ]:
# Custom CNN model
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(256 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = SimpleCNN(num_classes=len(idx_to_class)).to(DEVICE)
print(model)


In [ ]:
# Training loop
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

num_epochs = 15
best_val_acc = 0.0
patience = 5
wait = 0
best_model_path = WEIGHTS_DIR / 'best_model.pth'

history = {
    'train_loss': [],
    'val_loss': [],
    'val_acc': []
}

for epoch in range(1, num_epochs + 1):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)

    train_loss = running_loss / len(train_loader.dataset)

    model.eval()
    val_running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_loss = val_running_loss / len(val_loader.dataset)
    val_acc = correct / total

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    scheduler.step(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        wait = 0
        torch.save(model.state_dict(), best_model_path)
    else:
        wait += 1

    print(f'Epoch {epoch:02d}/{num_epochs} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f}')

    if wait >= patience:
        print('Early stopping triggered.')
        break

print('Best validation accuracy:', best_val_acc)
print('Best model saved to:', best_model_path)


In [ ]:
# Evaluation plots
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history['val_acc'], label='Val Accuracy')
plt.title('Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'training_curves.png', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# Confusion matrix and misclassified samples
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
model.eval()

all_true, all_pred = [], []
misclassified = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        outputs = model(images)
        preds = outputs.argmax(dim=1)

        all_true.extend(labels.cpu().numpy().tolist())
        all_pred.extend(preds.cpu().numpy().tolist())

for i in range(len(val_dataset)):
    image, true_label = val_dataset[i]
    with torch.no_grad():
        output = model(image.unsqueeze(0).to(DEVICE))
        pred_label = output.argmax(dim=1).item()
    if pred_label != true_label:
        misclassified.append((image, true_label, pred_label))

cm = confusion_matrix(all_true, all_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=False, cmap='Blues', xticklabels=[idx_to_class[i] for i in range(10)], yticklabels=[idx_to_class[i] for i in range(10)])
plt.title('Confusion Matrix (Validation)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.show()

# Plot 5 misclassified samples
n_show = min(5, len(misclassified))
fig, axes = plt.subplots(1, n_show, figsize=(3*n_show, 3))
if n_show == 1:
    axes = [axes]
for ax, (img_tensor, y_true, y_pred) in zip(axes, misclassified[:n_show]):
    img_np = img_tensor.permute(1,2,0).cpu().numpy()
    # unnormalize for display
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img_np = np.clip((img_np * std) + mean, 0, 1)
    ax.imshow(img_np)
    ax.set_title(f'True: {idx_to_class[y_true]}\nPred: {idx_to_class[y_pred]}')
    ax.axis('off')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'misclassified_samples.png', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# Public test preprocessing and submission

def preprocess_test(data_folder: str) -> pd.DataFrame:
    data_path = Path(data_folder)
    rows = []
    for img_path in sorted(data_path.iterdir()):
        if img_path.is_file() and img_path.suffix.lower() in IMG_EXTS:
            rows.append({'folder': str(data_path), 'file_name': img_path.name})
    return pd.DataFrame(rows)


class SatelliteTestDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = Path(row['folder']) / row['file_name']
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, row['file_name']


test_df = preprocess_test(str(TEST_DIR))
test_dataset = SatelliteTestDataset(test_df, transform=val_transform)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)

pred_rows = []
model.eval()
with torch.no_grad():
    for images, file_names in test_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy().tolist()
        for fname, pred_idx in zip(file_names, preds):
            pred_rows.append((fname, idx_to_class[pred_idx]))

pred_rows = sorted(pred_rows, key=lambda x: int(Path(x[0]).stem) if Path(x[0]).stem.isdigit() else x[0])
with open(SUBMISSION_PATH, 'w', encoding='utf-8') as f:
    for fname, pred_name in pred_rows:
        f.write(f'{fname},{pred_name}\n')

print('Saved submission to:', SUBMISSION_PATH)
print('Rows:', len(pred_rows))
